In [2]:
# 1. 드라이브 재마운트
from google.colab import drive
import os, shutil
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import unicodedata, json

drive.mount('/content/drive')

# 2. 이미 드라이브에 저장된 mission3_labels.zip을 코랩에 3초 만에 풀기
print("mission3_labels.zip 압축 해제 중...")
# 혹시 파일이 MyDrive 바로 아래에 있다면:
zip_in_drive = "/content/drive/MyDrive/mission3_labels.zip"

if os.path.exists(zip_in_drive):
    !unzip -q "/content/drive/MyDrive/mission3_labels.zip" -d /content/
elif os.path.exists("/content/mission3_labels.zip"):
    !unzip -q "/content/mission3_labels.zip" -d /content/
else:
    print("zip 파일을 찾는 중입니다...")
    import glob
    found = glob.glob("/content/drive/MyDrive/**/mission3_labels.zip", recursive=True)
    if found:
        !unzip -q "{found[0]}" -d /content/

print("전처리 시작")

# 3. 전처리 로직
TARGET_SYMPTOMS = ["고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란"]
SYMPTOM_TO_IDX = {sym: idx for idx, sym in enumerate(TARGET_SYMPTOMS)}

def normalize_text(text: str) -> str:
    return unicodedata.normalize("NFC", text)

def read_json_robust(path: Path) -> dict:
    for enc in ("utf-8-sig", "utf-8", "cp949"):
        try:
            with open(path, "r", encoding=enc) as f:
                return json.load(f)
        except Exception:
            continue
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return json.load(f)

def process_single_json(json_path: Path) -> dict:
    data = read_json_robust(json_path)
    utterances = data.get("utterances", [])
    texts = []
    if isinstance(utterances, list):
        for u in utterances:
            if isinstance(u, dict) and u.get("text"):
                texts.append(normalize_text(u["text"].strip()))
    full_text = " ".join(texts)

    raw_symptoms = data.get("symptom", [])
    if not isinstance(raw_symptoms, list):
        raw_symptoms = [raw_symptoms]

    filtered = []
    vector = [0] * len(TARGET_SYMPTOMS)
    for sym in raw_symptoms:
        if isinstance(sym, str):
            norm_sym = normalize_text(sym.strip())
            if norm_sym in SYMPTOM_TO_IDX:
                filtered.append(norm_sym)
                vector[SYMPTOM_TO_IDX[norm_sym]] = 1

    record = {
        "call_id": json_path.stem,
        "text": full_text,
        "symptoms": str(sorted(list(set(filtered)))),
        "label_vector": str(vector)
    }
    for idx, sym in enumerate(TARGET_SYMPTOMS):
        record[sym] = vector[idx]
    return record

def build_df(label_dir: str) -> pd.DataFrame:
    files = sorted(list(Path(label_dir).glob("*.json")))
    print(f"🔄 {label_dir} 처리 중 (총 {len(files)}개)...")
    return pd.DataFrame([process_single_json(f) for f in tqdm(files)])

# 4. 전처리 실행
train_df = build_df("/content/train/label")
val_df = build_df("/content/val/label")

# 혹시 train/label 경로가 /content/data/ 아래에 풀렸을 경우 대비
if len(train_df) == 0:
    train_df = build_df("/content/data/train/label")
    val_df = build_df("/content/data/val/label")

# 5. 구글 드라이브로 CSV 저장
drive_dir = "/content/drive/MyDrive"
train_df.to_csv(f"{drive_dir}/mission3_train.csv", index=False, encoding="utf-8-sig")
val_df.to_csv(f"{drive_dir}/mission3_val.csv", index=False, encoding="utf-8-sig")
print("\n mission3_train.csv, mission3_val.csv 생성")

# 6. 통계 요약
print("\n[9개 타겟 증상별 데이터 분포]")
print(pd.DataFrame({
    "Train 건수": train_df[TARGET_SYMPTOMS].sum(),
    "Val 건수": val_df[TARGET_SYMPTOMS].sum()
}))


Mounted at /content/drive
mission3_labels.zip 압축 해제 중...
⚠️ zip 파일을 찾는 중입니다...
전처리 시작
🔄 /content/train/label 처리 중 (총 29200개)...


100%|██████████| 29200/29200 [00:04<00:00, 6483.87it/s]


🔄 /content/val/label 처리 중 (총 3640개)...


100%|██████████| 3640/3640 [00:00<00:00, 4140.76it/s]



🎉 구글 드라이브에 mission3_train.csv, mission3_val.csv 생성

[9개 타겟 증상별 데이터 분포]
      Train 건수  Val 건수
고열        5186     678
구토        4463     537
두통        2905     374
복통        6771     838
어지러움      6334     774
열상        3465     463
오심        3341     429
전신쇠약      5310     651
호흡곤란      3927     492
